In [0]:
# Use the exact path from your Unity Catalog Volume
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("nullValue", "\\N") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/raw_data/circuit_csv/circuits.csv")

# This will display your F1 data
display(df)

In [0]:
%ls /Volumes/workspace/raw_data/circuit_csv/

In [0]:
df.printSchema()

In [0]:
# Filtering for circuits in the UK
uk_circuits_df = df.filter("location = 'Silverstone'")

# Showing the result
display(uk_circuits_df)

In [0]:
from pyspark.sql.functions import current_timestamp

# 1. Select and Rename in one flow
silver_df = df.select("circuitId", "circuitRef", "name", "location", "country", "lat", "lng") \
              .withColumnRenamed("circuitId", "circuit_id") \
              .withColumnRenamed("circuitRef", "circuit_ref")

# 2. Add the timestamp 'stamp'
silver_final_df = silver_df.withColumn("ingestion_date", current_timestamp())

# 3. View your work
display(silver_final_df)

In [0]:
# Saving the cleaned data as a permanent table in your Silver schema
silver_final_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_circuits")

In [0]:
%sql
-- Using the 3-tier namespace: catalog.schema.table
SELECT * FROM workspace.default.silver_circuits
LIMIT 10;